The profiling script performs deep inspection of Silver layer tables to generate a centralized audit trail. It calculates volumetric and quality metrics to ensure data reliability for downstream Gold layer processing.

**Key Features**
- Dynamic Table Discovery: Iterates through a JSON-defined list of datasets provided via Databricks widgets.
- Automated Audit Schema: Dynamically creates the audit schema if it does not exist within the Silver catalog.
- Volumetric Analysis: Captures total row counts, column counts, and unique record counts to verify deduplication success.
- Data Quality KPIs: Calculates total null counts and null_percent across all columns to flag data gaps.
- Incremental Auditing: Uses Delta Lake append mode with mergeSchema to maintain a historical log of data profiles over time.

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# --- 1. CONFIGURATION & WIDGET INITIALIZATION ---
# This line physically creates the widget if it doesn't exist
dbutils.widgets.text("datasets_json", '[]', "Datasets List (JSON Array)")

try:
    from schema_config import SILVER_CATALOG, SILVER_SCHEMA
    AUDIT_SCHEMA = "audit"
    PROFILE_TABLE = f"{SILVER_CATALOG}.{AUDIT_SCHEMA}.profile_summary"
except ImportError:
    # Fallback for testing if schema_config is missing
    SILVER_CATALOG, SILVER_SCHEMA = "data_silver", "silver"
    AUDIT_SCHEMA = "audit"
    PROFILE_TABLE = f"{SILVER_CATALOG}.{AUDIT_SCHEMA}.profile_summary"

def profile_silver_tables():
    """
    Profiles each table in the Silver layer and updates the audit.profile_summary table.
    """
    # Create Audit Schema if it doesn't exist
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.{AUDIT_SCHEMA}")
    
    # SAFE WIDGET RETRIEVAL
    try:
        datasets_raw = dbutils.widgets.get("datasets_json")
        dataset_list = json.loads(datasets_raw)
    except Exception as e:
        print(f"[ERROR] Could not read widget 'datasets_json'. Ensure it is a valid JSON array. Error: {e}")
        return

    if not dataset_list:
        print("[SKIP] No datasets provided in the widget.")
        return

    all_profiles = []

    for ds in dataset_list:
        table_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.{ds.lower()}"
        
        try:
            if not spark.catalog.tableExists(table_path):
                print(f"[SKIP] Table {table_path} not found.")
                continue
            
            print(f"[PROFILING] {ds}...")
            df = spark.table(table_path)
            
            # 1. Basic Counts
            row_count = df.count()
            col_count = len(df.columns)
            col_names = ", ".join(df.columns)
            
            # 2. Null Calculations
            null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
            null_data = df.select(null_counts_expr).collect()[0].asDict()
            total_nulls = sum(null_data.values())
            
            # 3. Precision Metrics
            null_percent = (total_nulls / (row_count * col_count)) * 100 if row_count > 0 else 0.0
            unique_count = df.dropDuplicates().count() #
            
            # 4. Profile Record
            all_profiles.append({
                "dataset_name": ds,
                "layer": "SILVER",
                "row_count": row_count,
                "column_count": col_count,
                "null_count": total_nulls,
                "null_percent": round(float(null_percent), 2),
                "unique_count": unique_count,
                "columns": col_names,
                "profile_timestamp": datetime.now()
            })
            
        except Exception as e:
            print(f"[ERROR] Could not profile {ds}: {str(e)}")

    if all_profiles:
        profile_df = spark.createDataFrame(all_profiles)
        
        # Requirement: Create if not exists, otherwise Append
        print(f"\n[UPDATING] {PROFILE_TABLE}...")
        profile_df.write.format("delta") \
                  .mode("append") \
                  .option("mergeSchema", "true") \
                  .saveAsTable(PROFILE_TABLE)
        
        print("[SUCCESS] Profiling data stored.")

# --- 2. EXECUTION ---
if __name__ == "__main__":
    profile_silver_tables()

In [0]:
%sql
select * from data_silver.audit.profile_summary